## **Business Focused Approach : How to Handle Imbalanced Data in Marketing**

### **Business Analytics in Telemarketing: Cost-Sensitive Analysis of Bank Campaigns Using Machine Learning**


### **Bank Telemarketing Campaign Analysis**

#### **Problem Statement**
Using a bank telemarketing dataset(45,211 customers, with 11.7% subscription rate). Will explore different approaches to handle class imbalance through a business lens. This notebook shows how to choose the right technique based on your specific business context and costs.

In [ ]:
## Import commonly used libraries
import pandas as pd 
import numpy as np  
import sidetable
import sklearn
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.metrics import precision_score, f1_score, recall_score, average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve, auc

: 

In [ ]:
## Dispaly Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
## Read the data
files = []
data_path = Path.cwd().parent.joinpath('data', 'raw')
for file in data_path.glob('*'):
    files.append(file.name)

print(files, end=" ")

In [ ]:
tele_df = pd.read_csv(data_path.joinpath('bank-full.csv'), sep=';')
tele_df.head()

In [ ]:
tele_df.shape

In [ ]:
(tele_df['y'].value_counts(normalize=True)*100).round(1)

__Bank Telemarketing Dataset Analysis :__

In [ ]:
# ==================================================
# PART 1: LOAD BANK TELEMARKETING DATASET
# ==================================================

# Title: Bank Marketing (without social/economic context)
# Source: https://archive.ics.uci.edu/ml/datasets/Bank+Marketing
# Citation : Moro, P. Cortez and P. Rita. A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems (2014)

def load_bank_telemarketing_data():
    """
    Load and preprocess the bank telemarketing dataset for term deposit subscription prediction.

    Returns:
    X, y, feature_names for modeling and analysis.
    """
    # Load the dataset
    tele_df = pd.read_csv(data_path.joinpath('bank-full.csv'), sep=';')
    
    print(f"Dataset shape: {tele_df.shape}")

    # Prepare features and target variable
    X_df = tele_df.drop('y', axis='columns')
    y_series = tele_df['y']

    # convert target variable to binary
    y = (y_series == 'yes').astype(int)

    print("Target Distribution:")
    print(f"No Subscription (0) : {(y == 0).mean() * 100:.1f}%")
    print(f"Subscription (1): {(y == 1).mean() * 100:.1f}%")

    # Handle categorical variables with Label encoding
    X_processed = X_df.copy()
    cat_columns = X_processed.select_dtypes(include=['object']).columns

    for col in cat_columns:
        le = LabelEncoder()
        X_processed[col] = le.fit_transform(X_df[col])

    X = X_processed.values
    feature_names = X_processed.columns.tolist()
    return X, y, feature_names

# Load the bank telemarketing data
X, y, feature_names = load_bank_telemarketing_data()
print("Loading Bank Telemarketing Dataset...")
print(f"Loaded Dataset : {len(y):,} samples with {len(feature_names)} features.")
print("\nDataset Characteristics:")
print(f"Total Samples: {len(y):,}")
print(f"Features : {len(feature_names)}")
print(f"No Subscription (0) : {(y == 0).sum():,} ({(y == 0).mean() * 100:.1f}%)")
print(f"Subscription (1): {(y == 1).sum():,} ({(y == 1).mean() * 100:.1f}%)")
print("This represents real scenario of bank telemarketing campaigns.\n")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ensure the split data as a numpy array
y_train = np.array(y_train)
y_test = np.array(y_test)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Set : {X_train_scaled.shape[0]:,} samples")
print(f"Test Set : {X_test_scaled.shape[0]:,} samples")
print(f"Training Subscription Cases: {(y_train==1).sum():,} ({(y_train==1).mean() * 100:.1f}%)")
print(f"Test Subscription Cases: {(y_test==1).sum():,} ({(y_test==1).mean() * 100:.1f}%)")


We're working with realistic sceanrio if bank telemarketing dataset from Protuguese telemarketing campaigns - 45211 contacts with 11.7% subscription rate for term deposits. This represent a business imbalanced dataset reflects real-world challenges in predicting term deposit subscriptions with rare positive cases.

The dataset contains 16 features including customer demographics(age, job, education, personal loan, housing loan, etc.,), campaign details(contact method, duration, previous, etc.). The 7.5:1 class imbalance makes this perfect for testing imbalance handling techniques.

Training : 36,168 samples (4,231 subscriptions)
Testing : 9,043 samples (1,058 subscriptions)

In [ ]:
# ===================================
# Visualize the Imbalance 
# ===================================

# Compute PCA with 2 components
pca = PCA(n_components=2, random_state=42)

# Seperate indices for Subscription and No-Subscription
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]

# Sample proportionally to maintain the true 88:12 ratio in visualization
# If we show 500 subscription cases, we should show ~3750 no-subscription(500 * 7.5) cases to maintain ratio
n_pos = 500
n_neg = int(n_pos * (len(neg_idx) / len(pos_idx)))

# Random sample maintaining proportions
pos_sample = np.random.choice(pos_idx, size= min(n_pos, len(pos_idx)), replace=False)
neg_sample = np.random.choice(neg_idx, size=min(n_neg, len(neg_idx)), replace=False)

viz_idx = np.concatenate([neg_sample, pos_sample])

X_viz = X_train_scaled[viz_idx]
y_viz = y_train[viz_idx]

# Apply pca
X_viz_pca = pca.fit_transform(X_viz)


fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# Scatter plot for pca
ax[0].scatter(X_viz_pca[y_viz ==0, 0], X_viz_pca[y_viz ==0, 1],
              c = 'grey', s=8, alpha=0.4, label=f'No Subscription (n={sum(y_viz==0)})')
ax[0].scatter(X_viz_pca[y_viz == 1, 0], X_viz_pca[y_viz == 1, 1],
            c='lightblue', alpha=0.8, s=12, label=f'Subscription (n={sum(y_viz==1)})')

ax[0].set_xlabel('PC 1')
ax[0].set_ylabel('PC 2')
ax[0].set_title('Original Imbalanced Dataset\n(True Proportional Representation)', fontsize=14, fontweight='bold', y=1.04)
ax[0].legend()

# Countplot for target variable
sns.countplot(
    data=tele_df, x='y', 
    order=tele_df['y'].value_counts().index, 
    palette = ['#ececec', '#3B8AEB90'],
    ax=ax[1])

ax[1].set_title('Distribution of Target Variable (y)', fontsize=14, fontweight='bold', y=1.04)
ax[1].set_xlabel('')
ax[1].set_ylabel('Number of Samples')
ax[1].set_xticks([0, 1], labels = ['No Subscription', 'Subscription'], rotation=0)

# Add percentage labels on bars
for i, v in enumerate(tele_df['y'].value_counts()):
    percentage = 100 * v / len(tele_df)
    ax[1].text(i, v + 200, f"{v:,} ({percentage:.1f}%)", ha='center', fontsize=10)

# Pie chart for target variable
tele_df['y'].value_counts().plot.pie(
    autopct='%1.1f%%', 
    colors = ['#ececec', "#3B8AEB90"], 
    startangle=90, labels = ['No Subscription', 'Subscription'],
    ax=ax[2])

ax[2].set_ylabel('')
ax[2].set_title('Proportion of Target Variable (y)', fontsize=14, fontweight='bold', y=1.10)

# Add layout adjustments
plt.tight_layout()
plt.subplots_adjust(wspace=0.3, hspace=0.3)
plt.show()



### **Baseline XGBoost**

In [ ]:
# ===============================================
# PART 3: BASELINE MODEL PERFORMANCE
# ===============================================

print("=== BASELINE MODEL: No Imbalance Handling ===")

# Train baseline XGBoost model
baseline_model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1
)
baseline_model.fit(X_train_scaled, y_train)

# Get predictions
y_pred_baseline = baseline_model.predict(X_test_scaled)
y_prob_baseline = baseline_model.predict_proba(X_test_scaled)[:,1]

# Comprehensive metrics function
def calculate_metrics(y_true, y_pred, y_prob, model_name):
    """
    Compute all relevant metrics for imbalanced classification problem
    """
    metrics = {}
    metrics['model'] = model_name
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1_score'] = f1_score(y_true, y_pred)
    metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    metrics['pr_auc'] = average_precision_score(y_true, y_prob)

    # balance accuracy
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp/ (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn/ (tn + fp) if (tn + fp) > 0 else 0
    metrics['balanced_accuracy'] = (sensitivity + specificity) / 2

    return metrics

baseline_metrics = calculate_metrics(y_test, y_pred_baseline, y_prob_baseline, "Baseline")
print("Baseline Model Performance:")
for key, value in baseline_metrics.items():
    if key != 'model':
        print(f"{key.replace('_', ' ').capitalize()}: {value:.4f}")

cm_baseline = confusion_matrix(y_test, y_pred_baseline)
print(f"\nConfusion Matrix:")
print(cm_baseline)

print(f"\nPrediction Analysis:")
print(f"Actually No Subscription: {sum(y_test == 0):,}, Predicted No Subscription: {sum(y_pred_baseline == 0):,}")
print(f"Actually Subscription: {sum(y_test == 1):,}, Predicted Subscription: {sum(y_pred_baseline == 1):,}")

# Calculate business impact
tn, fp, fn, tp = cm_baseline.ravel()
print(f"\nBusiness Impact:")
print(f"Missed Subscriptions (False Negatives): {fn} - Lost revenue!")
print(f"Wasted Marketing (False Positives): {fp} - Unnecessary costs")
print(f"Correctly Targeted Customers: {tp}")
     



**Analysis**

__Baseline XGBoost: Surprisingly Decent Performance__

The results are:

- **Predicted subscriptions (744):** This is 254 (false positives) + 490 (true positives) = 744 total predictions of "subscription"

- **Actually subscription (1058):** This is 568 (false negatives) + 490 (true positives) = 1,058 people who actually subscribed

- **Actually NO subscription (7985):** This is 7731 (true negatives) + 254 (false positives) = 7,985 people who actually didn't subscribe

Unlike extreme imbalanced scenarios, this real marketing data shows XGBoost handles moderate imbalance reasonably well:

- 54.4% F1-score, 46.3% subscription detection rate
- 254 wasted marketing efforts out of 9,043 customers
- Catches 490 out of 1,058 potential subscribers

__What this means in business terms:__

- __Wasted marketing efforts:__ The model predicted 744 customers would subscribe, but only 490 actually did. The other 254 customers received marketing calls/emails but said "no" - that's wasted time and money on uninterested prospects.

- __Missed subscriptions:__ Out of 1,058 customers who would actually subscribe if contacted, the model only identified 490 of them. The remaining 568 customers wanted the product but never got contacted - that's lost revenue.

The baseline shows typical imbalanced data bias - high precision (65.9%) but lower recall (46.3%). This conservative approach is **safe but costly** minimizes marketing waste but misses significant revenue opportunities. In simple terms: the model is very careful about who it recommends contacting (high precision) but misses many good prospects in the process (low recall). Or, simply (you avoid annoying uninterested customers, but you miss out on many who would have subscribed).



### **SMOTE Analysis ---- OVERSAMPLING APPROACH**

In [ ]:
# ===============================================
# PART 4: SMOTE APPROACH
# ===============================================

print("\n=== SMOTE OVERSAMPLING APPROACH ===")
from imblearn.over_sampling import RandomOverSampler 
from imblearn.over_sampling import SMOTE
# Apply SMOTE - but use a smaller sample to avoid memory issues
print("Applying SMOTE (this may take a moment with large datasets)...")
smote = SMOTE(random_state=42, k_neighbors=5)
ros = RandomOverSampler(random_state=42)
# For very large datasets, we might need to subsample before SMOTE
if len(X_train) > 30000:
    # If the training data is huge, take a smaller, balanced sample (20,000 rows) to make SMOTE faster and easier to demonstrate.
    print(f"Large dataset detected, using stratified method for subsample for SMOTE analysis.....")
    X_smote_sample, _, y_smote_sample, _ = train_test_split(
        X_train_scaled, y_train, train_size=20000, random_state=42, stratify=y_train
    )
else:
    # If the training data is small enough, use all of it.
    X_smote_sample, y_smote_sample = X_train_scaled, y_train

X_train_smote, y_train_smote = smote.fit_resample(X_smote_sample, y_smote_sample)

print(f"After SMOTE:")
print(f"Training samples: {len(X_train_smote):,}")
print(f"Non-subscribers: {sum(y_train_smote == 0):,} ({sum(y_train_smote == 0)/len(y_train_smote):.1%})")
print(f"Subscribers: {sum(y_train_smote == 1):,} ({sum(y_train_smote == 1)/len(y_train_smote)*100:.1f}%)")

# train model with SMOTE data
smote_model = xgb.XGBClassifier(
    objective = 'binary:logistic',
    random_state=42, 
    eval_metric='logloss',
    verbosity=0,
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1
)
# fit the model on smote samples
smote_model.fit(X_train_smote, y_train_smote)

# Get prediction
y_pred_smote = smote_model.predict(X_test_scaled)
y_prob_smote = smote_model.predict_proba(X_test_scaled)[:, 1]

smote_mertics = calculate_metrics(y_test, y_pred_smote, y_prob_smote, "SMOTE")

print("\nSMOTE Model Performance :")
for key, values in smote_mertics.items():
    if key != 'model':
        print(f"{key.replace('_', ' ').capitalize} : {values:.4f}")

cm_smote = confusion_matrix(y_test, y_pred_smote)
print(f"\nConfusion Matrix :")
display(cm_smote)

tn, fp, fn, tp = cm_smote.ravel()
print("\nBusiness Impact:")
print(f" - Missed Subscriptions : {fn}\n - Wasted Marketing : {fp}\n - Correctly Targeted : {tp}")

**Analysis**

__SMOTE Analysis: A Widely Used but Modest Performer__

The results are:

- **Predicted subscriptions (1,211):** This is 529 (false positives) + 682 (true positives) = 1,211 total predictions of "subscription"

- **Actually subscription (1,058):** This is 376 (false negatives) + 682 (true positives) = 1,058 people who actually subscribed

- **Actually NO subscription (7,985):** This is 7,456 (true negatives) + 529 (false positives) = 7,985 people who actually didn't subscribe

SMOTE balances classes by creating synthetic customer profiles, showing mixed results:

- F1-score improves to 60.1% (vs baseline 54.4%)
- Recall increases to 64.5% (catches 682 vs 490 subscribers)
- BUT precision drops to 56.3% (vs baseline 65.9%)
- More marketing contacts needed: 1,211 vs baseline's 744


__What this means in business terms:__

- __More Wasted marketing efforts:__ The model predicted 1,211 customers would subscribe, but only 682 actually did. The other 529 customers received marketing calls/emails but said "no" - that's more than double the baseline's wasted efforts (529 vs 254)..

- __Fewer Missed subscriptions:__ Out of 1,058 customers who would actually subscribe, SMOTE identified 682 of them vs baseline's 490. However, 376 potential subscribers still never got contacted.

__SMOTE vs Baseline comparison:__

- SMOTE finds 192 more subscribers (682 vs 490)
- But creates 275 more wasted marketing efforts (529 vs 254)
- Trade-off: For every additional subscriber found, SMOTE generates 1.4 extra wasted contacts(275/192)

__Why SMOTE shows limitations:__

1. Synthetic customers are interpolations between existing subscribers.
2. Doesn't explore new customer segments or behaviors.
3. Model learns from artificial patterns that may not generalize.
4. Improvement comes at the cost of marketing efficiency.



### **SMOTE Interpolation Problem: Visual Analysis**

In [ ]:
# ===============================================
# CRITICAL VISUALIZATION: SMOTE INTERPOLATION PROBLEM
# ===============================================

print("\n=== THE SMOTE INTERPOLATION PROBLEM ===")
print("Let's see exactly what SMOTE does - and why it can be problematic")
print("Using focused visualization to clearly show the interpolation pattern...")

# Create a CLEANER, more focused visualization showing SMOTE's interpolation
# Use the same PCA transformation fitted earlier for consistency
X_sample_pca = pca.transform(X_smote_sample) # norma/original samples
X_smote_pca = pca.transform(X_train_smote) # oversampling samples

# Identify original vs synthetic points
n_original = len(X_smote_sample)
original_mask = np.arange(len(X_train_smote)) < n_original
synthetic_mask = ~original_mask


In [ ]:
len(X_train_smote) - len(X_smote_sample)

In [ ]:
~(np.arange(len(X_train_smote)) < len(X_smote_sample))

The visualization clearly shows SMOTE only fills gaps between existing subscription patterns - it doesn't help identify completely new types of potential subscribers.

## **Class Weights: Tackling Imbalance with XGBoost’s Built-In Method**

In [ ]:
# ===================================================
# PART 5: CLASS WEIGHTs APPROACH
# ===================================================

print("\n ====== CLASS WEIGHTs APPROACH =====")

# compute optimal class weight
class_ratio = sum(y_train == 0) / sum(y_train ==1)
print(f"Optimal Class Ratio (No Subscription / Subscription) : {class_ratio:.1f}")

weighted_model = xgb.XGBClassifier(
    objective = 'binary:logistic',
    scale_pos_weight = class_ratio,
    random_state = 42,
    eval_metrics = 'logloss',
    verbosity = 0,
    max_depth= 6,
    n_estimators = 100,
    learning_rate=0.1
)

weighted_model.fit(X_train_scaled, y_train)

# Get predictions
y_pred_weighted = weighted_model.predict(X_test_scaled)
y_prob_weighted = weighted_model.predict_proba(X_test_scaled)[:, 1]

weighted_metrics = calculate_metrics(y_test, y_pred_weighted, y_prob_weighted, "Class Weighted")

print("\nClass Weighted Model Performance:")
for key, value in weighted_metrics.items():
    if key != 'model':
        print(f"{key.replace('_', ' ').capitalize()} : {value:.4f}")

cm_weighted = confusion_matrix(y_test, y_pred_weighted)
print("\nConfusion Matrix :")
print(cm_weighted)

tn, fp, fn, tp = cm_weighted.ravel()
print("\nBusiness Impact:")
print(f" - Missed Subscriptions : {fn}\n - Wasted Marketing : {fp}\n - Correctly Targeted : {tp}")


**Analysis**

__CLASS WEIGHTs: XGBoost's Built-in Imbalance Solution__

The results are:

- **Predicted subscriptions (2,078):** This is 1,158 (false positives) + 920 (true positives) = 2,078 total predictions of "subscription"

- **Actually subscription (1,058):** This is 138 (false negatives) + 920 (true positives) = 1,058 people who actually subscribed

- **Actually NO subscription (7,985):** This is 6,827 (true negatives) + 1,158 (false positives) = 7,985 people who actually didn't subscribe

XGBoost's scale_pos_weight parameter (set to 7.5) makes the model treat missed subscriptions as 7.5 times more costly than wasted marketing calls:

- Highest recall: 87% (920 out of 1,058 subscribers caught).
- Low precision: 44.3% (aggressive predictions create marketing waste).
- F1-score: 58.7% (lower than SMOTE due to precision penalty).
- Requires contacting 2,078 customers vs baseline's 744.

__What this means in business terms:__

- __Significantly more wasted marketing efforts:__ The model predicted 2,078 customers would subscribe, but only 920 actually did. The other 1,158 customers received marketing calls/emails but said "no" - that's 4.6x more waste than baseline (1,158 vs 254).

- __Fewer Missed subscriptions:__ Out of 1,058 customers who would actually subscribe, class weights identified 920 of them vs baseline's 490. However, 138 potential subscribers still never got contacted.

__Class Weights vs Baseline comparison:__

- Class weights finds 430 more subscribers (920 vs 490) -  an 88% improvement.
- But creates 904 more wasted marketing efforts (1,158 vs 254).
- Trade-off: For every additional subscriber found, SMOTE generates 2.1 extra wasted contacts(904/430).

__Class Weights vs SMOTE comparison:__

- Finds 238 more subscribers than SMOTE (920 vs 682).
- But creates 629 more wasted efforts than SMOTE (1,158 vs 529).
- More aggressive approach: casts wider net to catch more subscribers.

This approach focuses on capturing more subscriptions, even if it means less efficient marketing. It works best when subscriber lifetime value is much higher than campaign costs. No need for synthetic data—just smart parameter tuning that makes the model more aggressive in predicting subscriptions.


### **Threshold Tuning: Simple Optimization**

In [ ]:
# ===============================================
# PART 6: THRESHOLD TUNING
# ===============================================

print("\n=== THRESHOLD TUNING APPROACH ===")

# Find optimal threshold using baseline model probabilities
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_baseline)

# Calculate F1 scores for each threshold
f1_scores = []
for p, r in zip(precision, recall):
    if p + r == 0:
        f1_scores.append(0)
    else:
        f1_scores.append(2 * (p * r) / (p + r))

f1_scores = np.array(f1_scores)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5

print(f"Optimal threshold: {optimal_threshold:.3f} (default: 0.5)")
print(f"Expected F1 improvement: {f1_scores[optimal_idx]:.3f}")

# Apply optimal threshold
y_pred_threshold = (y_prob_baseline >= optimal_threshold).astype(int)

threshold_metrics = calculate_metrics(y_test, y_pred_threshold, y_prob_baseline, "Threshold Tuned")

print("\nThreshold Tuned Performance:")
for key, value in threshold_metrics.items():
    if key != 'model':
        print(f"{key.replace('_', ' ').capitalize()}: {value:.4f}")

cm_threshold = confusion_matrix(y_test, y_pred_threshold)
print(f"\nConfusion Matrix:")
print(cm_threshold)

tn, fp, fn, tp = cm_threshold.ravel()
print(f"\nBusiness Impact:")
print(f" - Missed Subscriptions: {fn}\n - Wasted Marketing: {fp}\n - Correctly Targeted: {tp}")
     

**Analysis**

__Threshold Tuning : Outstanding Results__

The results are:

- **Predicted subscriptions (1,479):** This is 672 (false positives) + 807 (true positives) = 1,479 total predictions of "subscription"

- **Actually subscription (1,058):** This is 251 (false negatives) + 807 (true positives) = 1,058 people who actually subscribed

- **Actually NO subscription (7,985):** This is 7,313 (true negatives) + 672 (false positives) = 7,985 people who actually didn't subscribe

Instead of default 0.5 threshold, optimizing for F1-score finds 0.261 works much better:

- Best F1-score: 63.6% (17% improvement over baseline).
- Balanced performance: 54.6% precision, 76.3% recall.
- Uses original model, just optimizes decision boundary.
- Requires contacting 1,479 customers vs baseline's 744.

__What this means in business terms:__

- __Moderate wasted marketing efforts:__ The model predicted 1,479 customers would subscribe, but only 807 actually did. The other 672 customers received marketing calls/emails but said "no" - that's 2.6x more waste than baseline (672 vs 254).

- __Well-balanced missed subscriptions:__ Out of 1,058 customers who would actually subscribe, threshold tuning identified 807 of them vs baseline's 490. However, 251 potential subscribers still never got contacted.

__Threshold Tuning vs Other Methods:__

- **vs Baseline:** Finds 317 more subscribers (807 vs 490) with 418 more wasted efforts (672 vs 254).
- **vs SMOTE:** Finds 125 more subscribers (807 vs 682) with 143 more wasted efforts (672 vs 529).
- **vs Class Weights:** Finds 113 fewer subscribers (807 vs 920) but 486 fewer wasted efforts (672 vs 1,158).
- **Trade-off ratio:** For every additional subscriber found vs baseline, threshold tuning generates 1.3 extra wasted contacts - the most efficient improvement.

Why this works so well:

- Leverages model's probability estimates more effectively.
- No training data modification required.
- Computationally trivial compared to SMOTE.
- Easily interpretable and adjustable for business needs.

This shows the power of proper threshold optimization - achieves the best balance between finding subscribers and controlling marketing waste with minimal effort.

### **Cost-Sensitive Optimization: Decisions Driven by Business Impact**

In [ ]:
# ===============================================
# PART 7: COST-SENSITIVE THRESHOLD
# We test different threshold values (0.1, 0.15, 0.2, etc. up to 0.9) and for each one,
# calculate the total business cost using our formula: (missed subscriptions × 200) + (wasted markerting x 10).
# The algorithm picks whichever threshold gives us the lowest total dollar cost, not the best statistical score.
# ===============================================

print("\n===== COST-SENSITIVE THRESHOLD OPTIMIZATION =====")

# In bank marketing: Missing potential subscriber = 200 lost lifetime value , Marketing to non-subscribers = 10 cost
cost_fn = 200  # Cost of missing a potential subscriber (lost lifetime value)
cost_fp = 10   # Cost of marketing to someone who won't subscribe

# find the threshold that mininmizes total cost
thresholds_test = np.arange(0.1, 0.9, 0.05)
costs = []

for thresh in thresholds_test:
    y_pred_temp = (y_prob_weighted >= thresh).astype(int)  # Use weighted model
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_temp).ravel()
    total_cost = fn * cost_fn + fp * cost_fp
    costs.append(total_cost)

optimal_cost_idx = np.argmin(costs)
optimal_cost_threshold = thresholds_test[optimal_cost_idx]

print(f"Cost-optimal threshold: {optimal_cost_threshold:.3f}")
print(f"Minimum total cost: ${costs[optimal_cost_idx]:,}")

y_pred_cost = (y_prob_weighted >= optimal_cost_threshold).astype(int)
cost_metrics = calculate_metrics(y_test, y_pred_cost, y_prob_weighted, "Cost-Optimized")

print("\nCost-Optimized Performance:")
for key, value in cost_metrics.items():
    if key != 'model':
        print(f"{key.replace('_', ' ').capitalize()}: {value:.4f}")

print("\nConfusion Matrix :")
cm_cost = confusion_matrix(y_test, y_pred_cost)
print(cm_cost)

tn, fp, fn, tp = cm_cost.ravel()
print(f"\nBusiness Impact:")
print(f"Missed Subscriptions: {fn} (${fn * cost_fn:,} lost revenue)")
print(f"Wasted Marketing: {fp} (${fp * cost_fp:,} cost)")
print(f"Total Cost: ${fn * cost_fn + fp * cost_fp:,}")


**Analysis :**

**Analysis**

__Cost-Sensitive Optimization__

**What This Approach Does:** Rather than optimizing for statistical measures like the F1-score, this approach focuses on minimizing real business costs.
    
- Missing a potential subscriber: $200 lost lifetime value.
- Wasted marketing contact: $10 cost per call/email.

The algorithm tests different thresholds and picks the one that minimizes total business cost, not statistical performance. It reflects real-world decisions where business impact matters more than academic metrics.

**NOTE :** Always try to make realistic estimates. In this case, I assumed a small deposit of about 5,000, with an average profit of 200(i.e., about 4% return).

The results are:

- **Predicted subscriptions (2,897):** This is 1,897 (false positives) + 1,000 (true positives) = 2,897 total predictions of "subscription"

- **Actually subscription (1,058):** This is 58 (false negatives) + 1,000 (true positives) = 1,058 people who actually subscribed

- **Actually NO subscription (7,985):** This is 6,088 (true negatives) + 1,897 (false positives) = 7,985 people who actually didn't subscribe

Cost-sensitive optimization finds threshold 0.250 delivers lowest total business cost:

- Highest recall: 94.5% (1,000 out of 1,058 subscribers found).
- Lowest precision: 34.5% (very aggressive predictions).
- Lowest total cost: 30,570 vs baseline's 116,140.
- Requires contacting 2,897 customers - nearly one-third of the entire test set.

__What this means in business terms:__

- __Maximum wasted marketing efforts:__ The model predicted 2,897 customers would subscribe, but only 1,000 actually did. The other 1,897 customers received marketing calls/emails but said "no" - that's 7.5x more waste than baseline (1,897 vs 254).

- __Minimal missed subscriptions:__ Out of 1,058 customers who would actually subscribe, threshold tuning identified 1,000 of them vs baseline's 490. Only, 58 potential subscribers still never got contacted -- the best performance.

__Cost-Sensitive vs Other Methods:__

- **vs Baseline:** Finds 510 more subscribers (1000 vs 490) with 1,643 more wasted efforts (1,897 vs 254).
- **vs SMOTE:** Finds 318 more subscribers (1000 vs 682) with 1,368 more wasted efforts (1,897 vs 529).
- **vs Class Weights:** Finds 80 more subscribers (1,000 vs 920) but 739 more wasted efforts (1,897 vs 1,158).
- **vs Threshold Tuning:** Finds 193 more subscribers (1,000 vs 807) but creates 1,225 more wasted efforts (1,897 vs 672).


__Business Cost Breakdown:__

- Lost revenue from 58 missed subscriptions: $11,600
- Marketing costs for 1,897 uninterested contacts: $18,970
- Total cost: 30,570(vs baseline's 116,140)

This approach captures maximum revenue but floods marketing teams with leads. Works well when missing a customer is costlier than extra marketing. However, it requires significant marketing capacity to handle the volume of contacts.


### **Business Cost Comparison Across All Approaches**

**Cost Assumptions (hypothetical for demonstration):**

- Missing a potential subscriber: $200 lost lifetime value
- Wasted marketing contact: $10 cost per call/email

| Approach | Misses Suscriptions | Lost Revenue | Wasted Marketing | Marketing Cost | Total Cost | Subscribers Found | Detection Rate |
|-------|-------|-------|-------|-------|-------|-------|-------|
| Baseline  | 568  | $113,600  | 254  | $2,540  | $116,140  | 490  | 46.3%  |
| SMOTE  | 376  | $75,200  | 529  | $5,290  | $80,490  | 682  | 64.5%  |
| Class Weighted  | 138  | $27,600  | 1,158  | $11,580  | $39,180  | 920  | 87.0%  |
| Threshold Tuned  | 251  | $50,200  | 672  | $6,720  | $56,920  | 807  | 76.3%  |
| Cost-Optimized | 58      |  $11,600      | 1,897     | $18,970      | $30,570      |   1,000     |   94.5%    |

**Key Business Insights:**

**Lowest Total Cost:** Cost-Optimized ($30,570) - despite highest marketing waste, minimizes expensive missed subscriptions.

**Highest Total Cost:** Baseline ($116,140) - conservative approach costs the most due to massive revenue loss.

**Best Detection Rate:** Cost-Optimized (94.5%) - finds nearly all potential subscribers.

**Cost Savings:** Cost-Optimized saves $85,570 compared to Baseline approach.

**The Business Logic:**

- Missing a subscriber costs $200 (lifetime value).
- Wasted marketing contact costs $10 (call/email expense).
- Therefore, it's better to waste 10 marketing contacts than miss 1 subscriber.

**Strategic Implications:**

- High-value subscriptions: Use Cost-Optimized (lowest total cost).
- Limited marketing capacity: Use Threshold Tuned (best balance).
- Risk-averse campaigns: Use SMOTE (moderate approach).
- **Avoid:** Baseline approach (highest cost due to missed revenue).

This table illustrate why business-focused optimization outperforms statistical metrics - the approach with the lowest F1-score (Cost-Optimized) delivers the best business outcome by understanding the true cost structure.